# Entregável 2 — RecFair workflow (relatório)

> **Projeto:** RecFair — recomendação com contrato utilidade + justiça  
> **Arquitetura vigente:** `workflow` · `prompt_version=v2`  
> **Baseline:** `baseline` · `prompt_version=v1` (reexecutado na mesma régua)  
> **Data:** 13/09/2026

Notebook **somente relatório**: importa o pacote `recfair/` e `eval/`. Sem lógica de grafo, scoring ou verify duplicada.

# A. Estrutura herdada do Entregável 1

- Schema: `recfair.schemas.output.RecFairOutput`
- Golden-set: `data/golden/cases.json` (T01–T30 imutáveis + T31–T38 novos)
- Verify: `eval.verify.verify_case`
- Runner: `eval.runner.run_eval`
- Dados E2: `tb_claims` (fichas boticario.com.br) + `tb_inventory` — manifest em `data/claims_manifest.json`

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from eval.fingerprint import golden_revision, load_cases
from eval.gold import gold_for
from eval.report import (
    build_arch_comparison_table,
    build_results_table,
    render_arch_comparison,
    render_comparison_report,
    render_metrics_panel,
)
from eval.runner import run_eval
from eval.verify import verify_case
from recfair.config import apply_dotenv, ensure_google_api_key, model_version
from recfair.observability.run_record import git_sha
from recfair.schemas.output import RecFairOutput

apply_dotenv()
print("chave:", ensure_google_api_key())
cases = load_cases()
print(len(cases), "casos | golden_revision =", golden_revision(cases))
print("modelo:", model_version(), "| git:", git_sha())
manifest_path = Path("data/claims_manifest.json")
if manifest_path.is_file():
    claims_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"tb_claims: {len(claims_manifest)} SKUs com source_url")

chave: GOOGLE_API_KEY
38 casos | golden_revision = 15f3ed6986de9ce9
modelo: gemini-3.5-flash-lite | git: 69dab185e84b


/home/andersonbr/estudos/unicamp-llm-agents/recfair/venv-recfair/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## A. Estrutura herdada do E1

Reutilizaremos a arquitetura E1 (baseline)

# B. Hipótese arquitetural

1. **Limitação E1:** stuffing de catálogo+vendas no prompt falha em agregação determinística (janela 7d, ordem), filtros de preço/estoque, claims e memória multi-turn.
2. **Mecanismo:** workflow LangGraph com `parse_intent` (LLM) + pipeline de scoring determinístico em 7 passos sobre SQLite; `MemorySaver` + `thread_id`.
3. **Espera melhorar:** T05/T12 (ordem), T17/T20 (preço), T23 (estoque), T16/T36 (claims), T31–T33 (memória), T37 (launch).
4. **Espera piorar:** latência e chamadas LLM (parse_intent por turno); complexidade operacional; T26–T30 (guardrails) permanecem gap.

# C. Arquitetura da v2

- [x] **workflow determinístico** — LLM só em `parse_intent`; scoring fixo via `scoring/engine.py`
- [ ] agente ReAct
- [ ] arquitetura híbrida

Executar no terminal: `make chat ARCH=workflow` (ou `make chat` — vigente). Comandos: `/trace`, `/reset`, `/arch`.

In [2]:
from recfair.graphs.registry import list_architectures

print("Arquiteturas registradas:", list_architectures())

Arquiteturas registradas: ['baseline', 'workflow']


# D. Contexto e memória

Casos T31–T33 são multi-turn com `thread_id=case_id`. Sem checkpoint, turno 2 perde categoria/marca refinada no turno 1.

**Isolamento:** cada `thread_id` no `MemorySaver` do workflow. CLI gera UUID por sessão; `/reset` limpa checkpoint.

In [3]:
# Demo memória: T33 — requer GOOGLE_API_KEY e arch=workflow
from recfair.graphs import workflow as workflow_mod

case_t33 = next(c for c in cases if c["id"] == "T33")
tid = "demo-t33"
workflow_mod.reset_checkpoint(tid)
for turn in case_t33["turns"]:
    out, metrics = workflow_mod.run(turn, thread_id=tid)
    skus = [i.sku for i in out.items] if out.items else []
    print(turn[:50], "...", "->", skus)

Quais os produtos de cabelo da Match mais vendidos ... -> ['F3P9W2', 'L6K1C8', '2Y8N4T', 'R5B7Q3', '9C4M1H']
Me mostra os mesmos de novo. ... -> []


# E. Ferramentas e integração externa

**Decisão:** ferramentas locais SQLite (`tb_catalogo`, `tb_vendas`, `tb_claims`, `tb_inventory`). MCP não adotado — dados já no pacote, sem latência de rede nem superfície de ataque extra.

| Tool | Entrada | Saída |
| :--- | :--- | :--- |
| filter_by_category_brand | category, brand? | pool SKU |
| exclude_stock_and_price | pool, max_price? | pool filtrado |
| score_claims | pool, claim_terms | +2 pontos |
| score_brand_diversity | pool | +1 representante |
| add_promo_launch | pool | +1 launch/promo |
| rank_by_sales_tiebreak | pool pontuado | ranking |
| assemble_top5 | ranking | Top 5 |

# F. Comparação baseline × workflow (mesma régua)

Gabarito unificado via `scoring/engine.py`. Baseline E1 **não foi alterado**; só a expectativa evoluiu.

**Correções de gabarito (T01–T30):** casos com promo/estoque/preço/claims/diversidade passam a usar o engine; diferenças vs E1 aparecem na coluna `diff` abaixo.

In [4]:
manifest_baseline = run_eval(arch="baseline", persist=True)

Unexpected argument 'thinking_level' provided to ChatGoogleGenerativeAI. Did you mean: 'thinking_budget'?
/home/andersonbr/estudos/unicamp-llm-agents/recfair/recfair/graphs/baseline.py:139: UserWarning: WARNING! thinking_level is not default parameter.
                thinking_level was transferred to model_kwargs.
                Please confirm that thinking_level is what you intended.
  structured = _get_structured_llm()


In [5]:
manifest_workflow = run_eval(arch="workflow", persist=True)

In [6]:
assert manifest_baseline["golden_revision"] == manifest_workflow["golden_revision"]
print("golden_revision:", manifest_baseline["golden_revision"])

golden_revision: 15f3ed6986de9ce9


In [15]:
_E1_LABELS = {
    "arch_title": "Comparação baseline × workflow (E1)",
    "arch_subtitle": "golden_revision compartilhado",
    "metrics_subtitle": "Run baseline E1",
    "rate_restrict_hint": "Escopo restrito Baseline E1",
    "rate_overall_hint": "Escopo completo Workflow E2",
    "results_title": "Baseline × gabarito (golden-set)",
}
_E2_LABELS = {
    "arch_title": "Comparação baseline × workflow (E2)",
    "arch_subtitle": "golden_revision compartilhado",
    "metrics_subtitle": "Run workflow E2",
    "rate_restrict_hint": "Escopo restrito Baseline E1",
    "rate_overall_hint": "Escopo completo Workflow E2",
    "results_title": "Workflow × gabarito (golden-set)",
}

In [16]:
display(
    HTML(
        render_arch_comparison(
            manifest_baseline,
            manifest_workflow,
            title=_E2_LABELS["arch_title"],
            subtitle=_E2_LABELS["arch_subtitle"],
            left_label="baseline",
            right_label="workflow",
            metric_prefix="e2",
        )
    )
)

métrica,baseline,workflow,delta
Restrito (S_*),0.1739,0.6957,0.5218
Scoring (T34–T38),0.0,0.6,0.6
Memória (T31–T33),0.0,0.0,0.0
Geral,0.1579,0.7368,0.5789
Latência mediana (s),1.98,0.81,-1.17
Tokens entrada média,29423.6,424.4,-28999.2
Tokens saída média,454.9,52.1,-402.8
Chamadas LLM,41.0,41.0,0.0
Chamadas tools,0.0,217.0,217.0
Custo est. (USD),0.378643,0.009531,-0.37


In [17]:
display(
    HTML(
        render_metrics_panel(
            manifest_baseline["resumo"],
            metric_prefix="e1",
            subtitle=_E2_LABELS["metrics_subtitle"],
            rate_restrict_hint=_E2_LABELS["rate_restrict_hint"],
            rate_overall_hint=_E2_LABELS["rate_overall_hint"],
        )
    )
)

display(
    HTML(
        render_metrics_panel(
            manifest_workflow["resumo"],
            metric_prefix="e2",
            subtitle=_E2_LABELS["metrics_subtitle"],
            rate_restrict_hint=_E2_LABELS["rate_restrict_hint"],
            rate_overall_hint=_E2_LABELS["rate_overall_hint"],
        )
    )
)

e1_rate_restrictEscopo restrito Baseline E1,4/23 (17.4%)
e1_rate_overallEscopo completo Workflow E2,6/38 (15.8%)
Latência mediana,1.98 s
Latência média,2.03 s
Chamadas LLM,41
Tokens entrada / saída,"1,118,095 / 17,286"
Custo estimado (USD),$0.3786


e2_rate_restrictEscopo restrito Baseline E1,16/23 (69.6%)
e2_rate_overallEscopo completo Workflow E2,28/38 (73.7%)
Latência mediana,0.81 s
Latência média,0.87 s
Chamadas LLM,41
Tokens entrada / saída,"15,703 / 1,928"
Custo estimado (USD),$0.0095


In [18]:
tabela_wf = build_results_table(manifest_workflow["records"], output_column="workflow")
display(
    HTML(
        render_comparison_report(
            tabela_wf,
            "status_final",
            title=_E2_LABELS["results_title"],
            code_columns=frozenset({"workflow", "gabarito"}),
        )
    )
)

caso,tipo caso,tipo teste,status final,workflow,gabarito,diff,motivo erro
T01,normal,restrito,sucesso,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,igual ao gabarito,—
T02,paráfrase,restrito,sucesso,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,igual ao gabarito,—
T03,composto,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,igual ao gabarito,—
T04,normal,restrito,sucesso,24A51X → Z5C1R8 → X1Q8V3 → 7H5A2E → K8M2Q1,24A51X → Z5C1R8 → X1Q8V3 → 7H5A2E → K8M2Q1,igual ao gabarito,—
T05,janela_7d,restrito,sucesso,7K2N9A → 2M7K4F → 8V4C6N → H3L9Q1 → P1T8R5,7K2N9A → 2M7K4F → 8V4C6N → H3L9Q1 → P1T8R5,igual ao gabarito,—
T06,informação ausente,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T07,informação ausente,restrito,sucesso,abstention · unknown_brand,abstention · unknown_brand,—,—
T08,ambíguo,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T09,ambíguo,restrito,erro,7K2N9A → Q4H8L2 → 3R1B6M → 5J8P2X → W9C5TD,abstention · missing_category,—,"RF-04/05: deveria abster, obteve status=recommendation"
T10,fora de escopo,restrito,sucesso,abstention · unknown_category,abstention · unknown_category,—,—


# G. Modos de falha agênticos

| Modo | Sintoma | Mitigação E2 |
| :--- | :--- | :--- |
| parse_intent errado | categoria/marca/preço incorretos | prompt v2 + merge com session_intent |
| abstain indevido | perde recomendação válida | scoring só após intent explícito |
| perda de memória | turno 2 ignora refinamento | MemorySaver + thread_id |
| recursion_limit | loop no grafo | limite 12 no invoke |
| PII/injection (T26–T30) | vazamento ou obediência | **gap E3** — sem sanitize_pii |

Exemplo de `ScoreTrace` (T36 — claim anticaspa):

In [11]:
from recfair.scoring.engine import score_recommendation
from recfair.scoring.case_intent import intent_from_case

case_t36 = next(c for c in cases if c["id"] == "T36")
result = score_recommendation(intent_from_case(case_t36))
for entry in result.trace.to_dicts():
    if entry.get("sku") in {"H8Q3N1", "-"} or entry.get("action") == "bonus":
        print(entry)

{'step': 'filter_by_category_brand', 'sku': 'H8Q3N1', 'action': 'pool', 'reason': 'category=cabelos', 'points_delta': 0, 'points_total': 0}
{'step': 'exclude_stock_and_price', 'sku': 'H8Q3N1', 'action': 'pool', 'reason': 'in stock', 'points_delta': 0, 'points_total': 0}
{'step': 'score_claims', 'sku': 'H8Q3N1', 'action': 'bonus', 'reason': "match claim 'anticaspa'", 'points_delta': 2, 'points_total': 2}
{'step': 'score_brand_diversity', 'sku': 'F3P9W2', 'action': 'bonus', 'reason': 'brand representative (Match)', 'points_delta': 1, 'points_total': 1}
{'step': 'score_brand_diversity', 'sku': 'H8Q3N1', 'action': 'bonus', 'reason': 'brand representative (Malbec)', 'points_delta': 1, 'points_total': 3}
{'step': 'score_brand_diversity', 'sku': 'A8T3K5', 'action': 'bonus', 'reason': 'brand representative (Cuide-se Bem)', 'points_delta': 1, 'points_total': 1}
{'step': 'add_promo_launch', 'sku': '3G7P2W', 'action': 'bonus', 'reason': 'is_promo=true', 'points_delta': 1, 'points_total': 1}
{'ste

# H. Análise arquitetural

**Hipótese confirmada?** [Preencher após executar as células F com API key.]

**Pergunta obrigatória (candidato E3):** guardrails de PII (`sanitize_pii`) antes do `parse_intent` + auditoria de fairness nas recomendações — o workflow melhora utilidade, mas T26–T30 mostram que entrada não confiável ainda atravessa o LLM sem filtro.

---

### Checklist de entrega E2

- [x] Estrutura herdada (seção A)
- [x] Hipótese antes da implementação (B)
- [x] Workflow com estado, >1 etapa, tool real, roteamento, recursion_limit (C)
- [x] Memória demonstrada (D)
- [x] Integração justificada (E)
- [ ] Comparação baseline reexecutado — executar células F
- [x] Modos de falha documentados (G)
- [ ] Análise final preenchida (H)